# 🚀 **PPO** Agent playing for **LunarLander-v3**

### Mounting Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Packages & Libraries

In [ ]:
!apt install swig cmake

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
!pip install huggingface_sb3 stable_baselines3 "gymnasium[box2d]" swig imageio

In [ ]:
import gymnasium as gym

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback

import imageio

### Environment Testing

In [ ]:
# LunarLander-v3 Env
env = gym.make("LunarLander-v3")

# Reset Env
observation, info = env.reset()

for _ in range(20):
  # Random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # Do this action in the environment and get
  # next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated (in our case we land, crashed) or truncated (timeout)
  if terminated or truncated:
      # Reset the environment
      print("Environment is reset")
      observation, info = env.reset()

env.close()

In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("LunarLander-v3")
env.reset()
print("Observation Space", "="*60, sep="\n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample())  # Random observation

In [ ]:
print("Action Space", "="*60, sep="\n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())  # Take a random action

### Define Env, Policy Model

In [ ]:
# Env Params
env_id = "LunarLander-v3"
vec_n_envs = 16

# PPO Core Params
policy_id = "MlpPolicy"
learning_rate = 3e-4       # good default, can try 1e-4 for stability
n_steps = 2048             # rollout length
batch_size = 128           # much larger than 16, improves gradient signal
n_epochs = 10              # more passes per batch
gamma = 0.99               # reward discount
gae_lambda = 0.95          # GAE lambda (0.95 often better than 0.98 here)
clip_range = 0.2           # PPO clip
ent_coef = 0.01            # entropy bonus (can decay over time)
max_grad_norm = 0.5        # gradient clipping
vf_coef = 0.5              # value function loss coef
target_kl = None           # disable by default
verbose = 1

# Callback & Training Params
eval_freq = 10000
total_timesteps = 2_500_000

In [ ]:
# Train Env (with normalization)
train_env = make_vec_env(env_id, n_envs=vec_n_envs)
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.)

# Eval Env (single env for evaluation)
eval_env = make_vec_env(env_id, n_envs=1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=True, clip_obs=10.)
# Important: use a separate copy (don't share stats between train/eval)


# Agent
model = PPO(
    policy=policy_id,
    env=train_env,
    learning_rate=learning_rate,
    n_steps=n_steps,
    batch_size=batch_size,
    n_epochs=n_epochs,
    gamma=gamma,
    gae_lambda=gae_lambda,
    ent_coef=ent_coef,
    clip_range=clip_range,
    max_grad_norm=max_grad_norm,
    vf_coef=vf_coef,
    verbose=verbose,
)

# Eval Callback
log_path = "logs/"
best_model_path = "best_model/"

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=best_model_path,
    log_path=log_path,
    eval_freq=eval_freq,
    deterministic=True,
    render=False
)

### Training & Saving

In [ ]:
# Train
model.learn(
    total_timesteps=total_timesteps,
    callback=eval_callback,
    progress_bar=True
)

model.save("ppo_lunarlander_v3_25hk_v01")
train_env.save("vecnormalize.pkl")

### Testing

In [ ]:
# Load trained model
model = PPO.load("ppo_lunarlander_v3_25hk_v01")

# Recreate environment
eval_env = gym.make("LunarLander-v3", render_mode="human")  # human for watching
eval_env = DummyVecEnv([lambda: eval_env])

# Load VecNormalize stats (disable training)
eval_env = VecNormalize.load("vecnormalize.pkl", eval_env)
eval_env.training = False
eval_env.norm_reward = False

# Run a few episodes
obs = eval_env.reset()
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = eval_env.step(action)
    if done:
        obs = eval_env.reset()

In [ ]:
"""
# Enjoy trained agent
vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")
"""

### Evaluation

In [ ]:
# Evaluate Policy
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

### Record-Video

In [ ]:
# Record Video
video_path = "ppo-LunarLander-v3-lander.mp4"
env_video = gym.make(env_id, render_mode="rgb_array")
obs, _ = env_video.reset()
frames = []

for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env_video.step(action)
    frames.append(env_video.render())
    if done or truncated:
        obs, _ = env_video.reset()

imageio.mimsave(video_path, frames, fps=30)
print(f"Video saved to {video_path}")

### Push to HF-Hub

In [ ]:
notebook_login()
!git config --global credential.helper store

In [ ]:
repo_id = "AminVilan/ppo-LunarLander-v3"

env_id = "LunarLander-v3"

model_name = "v01-ppo-LunarLanderV3"
model_architecture = "PPO"

commit_message = "Upload PPO LunarLander-v3 trained agent, Version-01, 2.5M."

In [ ]:
eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=True, clip_obs=10.)

package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message,
)

## The End 🐳